# Dataset Preparation
**This notebook will not be shared with the students.**
Prepare data from the `hseb-benchmark/msmarco` HuggingFace dataset for the home assignment.

In [1]:
%cd ../

/Users/jakubzovak/Documents/advanced_ai_driven_search


In [2]:
%load_ext autoreload
%autoreload 2

## Setup

In [3]:
from typing import cast
import random

from datasets import load_dataset
from datasets.dataset_dict import DatasetDict
from fastembed import LateInteractionTextEmbedding, TextEmbedding

## Load Models

Model to precompute vector embeddings.

In [4]:
embedding_model = TextEmbedding('sentence-transformers/all-MiniLM-L6-v2')

Make work with the ColBERT model easier for students by precomputing multi vector embeddings.

In [5]:
multi_vector_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

## Load Dataset
Load the existing HSBE dataset from HuggingFace.

In [5]:
# Queries need to be created from running the retrieval pipeline
# query = load_dataset("hseb-benchmark/msmarco", "query-all-MiniLM-L6-v2-100K")

corpus: DatasetDict = cast(DatasetDict, load_dataset("hseb-benchmark/msmarco", "corpus-all-MiniLM-L6-v2-100K")) 

## Additional Dataset Processing
Add multi-vector embeddings with colbert model to the documents. Also, add metadata information and tenant identifier.

In [6]:
def add_multi_vector_embedding(example):
    """Embed text using multi_vector_model and add to feature 'multi_vector_embedding'."""
    multi_vector_embedding = list(multi_vector_model.embed([example["text"]]))[0].tolist()
    
    example['multi_vector_embedding'] = multi_vector_embedding
    return example

In [7]:
# Remove the "tag" feature and add "groups" feature (as a list)
# Each example can belong to multiple groups randomly based on probabilities:
# group_1: 50%, group_2: 55%, ..., group_10: 95%

# Set seed for reproducibility
random.seed(42)

def assign_groups(example):
    """Assign groups randomly based on probabilities. Each example can have multiple groups."""
    groups = []
    
    # Group probabilities
    group_probabilities = {
        "group_1": 0.50,
        "group_2": 0.55,
        "group_3": 0.60,
        "group_4": 0.65,
        "group_5": 0.70,
        "group_6": 0.75,
        "group_7": 0.80,
        "group_8": 0.85,
        "group_9": 0.90,
        "group_10": 0.95,
    }
    
    # Randomly assign each group based on its probability
    for group_name, probability in group_probabilities.items():
        if random.random() < probability:
            groups.append(group_name)
    
    # Add the groups field (as a list)
    example['groups'] = groups
    return example

# Apply the transformation to each split in the corpus
for split_name in corpus.keys():
    # Add groups field
    corpus[split_name] = corpus[split_name].map(assign_groups)
    # Remove tag field
    corpus[split_name] = corpus[split_name].remove_columns(['tag'])
    corpus[split_name] = corpus[split_name].map(add_multi_vector_embedding)

Parameter 'function'=<function add_multi_vector_embedding at 0x12a485bc0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

## Save Modified Dataset Locally

In [8]:
corpus.save_to_disk("./corpus-all-MiniLM-L6-v2-100K-groups-multi-vector")

Saving the dataset (0/17 shards):   0%|          | 0/100000 [00:00<?, ? examples/s]

## Load & Take Subset of Documents

In [4]:
documents_dataset: DatasetDict = DatasetDict.load_from_disk("corpus-all-MiniLM-L6-v2-100K-groups-multi-vector")

Loading dataset from disk:   0%|          | 0/17 [00:00<?, ?it/s]

In [ ]:
# Create a subset with exactly half of the documents
documents_dataset_50k = DatasetDict()

for split_name in documents_dataset.keys():
    # Get the total number of examples in this split
    total_size = len(documents_dataset[split_name])
    half_size = total_size // 2
    
    # Select first half of the dataset
    documents_dataset_50k[split_name] = documents_dataset[split_name].select(range(half_size))
    
    print(f"{split_name}: {total_size} -> {half_size} examples")

# Save the 50K subset
documents_dataset_50k.save_to_disk("./corpus-all-MiniLM-L6-v2-50K-groups-multi-vector")
print("\nDataset saved as 'corpus-all-MiniLM-L6-v2-50K-groups-multi-vector'")
# TODO: Upload the dataset to the Hugging Face dataset


train: 100000 -> 50000 examples


Saving the dataset (0/9 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]


Dataset saved as 'corpus-all-MiniLM-L6-v2-50K-groups-multi-vector'


In [5]:
documents_dataset_50k_rating: DatasetDict = DatasetDict.load_from_disk("./data/corpus-all-MiniLM-L6-v2-50K-groups-multi-vector")

In [7]:
# Add "rating" feature with random float values from 0.0 to 1.0
random.seed(42)

def add_rating(example):
    """Add a random rating between 0.0 and 1.0"""
    example['rating'] = random.random()
    return example

# Apply the transformation to each split
for split_name in documents_dataset_50k_rating.keys():
    documents_dataset_50k_rating[split_name] = documents_dataset_50k_rating[split_name].map(add_rating)

print("Rating feature added successfully!")


Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Rating feature added successfully!


In [8]:
documents_dataset_50k_rating.save_to_disk("./data/corpus-all-MiniLM-L6-v2-50K-groups-multi-vector-rating")

Saving the dataset (0/9 shards):   0%|          | 0/50000 [00:00<?, ? examples/s]

## Queries Preprocessing

In [ ]:
import random

query_dataset: DatasetDict = cast(DatasetDict, load_dataset("hseb-benchmark/msmarco", "query-all-MiniLM-L6-v2-100K"))

# Additional preprocessing of the queries
# 1. Take first 100 queries
# 2. Remove specified columns
# 3. Add "filters" column with distribution: 30% empty, 40% one value, 30% two values

# Set seed for reproducibility
random.seed(42)

# Process each split in the query dataset
for split_name in query_dataset.keys():
    # Take first 100 queries
    query_dataset[split_name] = query_dataset[split_name].select(range(100))
    
    # Remove specified columns
    columns_to_remove = [
        'embedding', 
        'results_10_docs', 
        'results_10_scores', 
        'results_90_docs', 
        'results_90_scores', 
        'results_100_docs', 
        'results_100_scores'
    ]
    query_dataset[split_name] = query_dataset[split_name].remove_columns(columns_to_remove)
    
    # Add filters column
    def add_filters(example, idx):
        """Add filters based on distribution: 30% empty, 40% one value, 30% two values"""
        available_groups = [
            "group_1", "group_2", "group_3", "group_4", "group_5",
            "group_6", "group_7", "group_8", "group_9", "group_10"
        ]
        
        # Determine number of filters based on distribution
        rand_val = random.random()
        if rand_val < 0.30:  # 30% empty
            filters = []
        elif rand_val < 0.70:  # 40% one value (30% + 40%)
            filters = [random.choice(available_groups)]
        else:  # 30% two values
            filters = random.sample(available_groups, 2)
        
        example['filters'] = filters
        return example
    
    query_dataset[split_name] = query_dataset[split_name].map(add_filters, with_indices=True)

DatasetDict({
    train: Dataset({
        features: ['text', 'id', 'filters'],
        num_rows: 100
    })
})

In [4]:
query_dataset.save_to_disk("./query-all-MiniLM-L6-v2-100-filters")

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

## Queries Embeddings
Precompute embeddings for the queries.

In [7]:
query_dataset: DatasetDict = DatasetDict.load_from_disk("query-all-MiniLM-L6-v2-100-filters")

In [8]:
# Examples of computing embeddings
# query_dense_embedding: list[float] = list(embedding_model.query_embed([query_text]))[0].tolist()
# query_multi_vector_embedding: list[list[float]] = list(multi_vector_model.embed([query_text]))[0].tolist()

# Add embeddings to the query dataset
def add_query_embeddings(example):
    """Compute and add both dense and multi-vector embeddings for query text."""
    # Compute dense embedding using query_embed
    dense_embedding = list(embedding_model.query_embed([example["text"]]))[0].tolist()
    
    # Compute multi-vector embedding
    multi_vector_embedding = list(multi_vector_model.embed([example["text"]]))[0].tolist()
    
    # Add both embeddings to the example
    example['embedding'] = dense_embedding
    example['multi_vector_embedding'] = multi_vector_embedding
    
    return example

# Apply the transformation to each split in the query dataset
for split_name in query_dataset.keys():
    query_dataset[split_name] = query_dataset[split_name].map(add_query_embeddings)

Parameter 'function'=<function add_query_embeddings at 0x14647ac00> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
query_dataset.save_to_disk("query-all-MiniLM-L6-v2-100-filters-embedded")
# TODO: Upload the dataset to the Hugging Face dataset

Saving the dataset (0/1 shards):   0%|          | 0/100 [00:00<?, ? examples/s]

## Upload Final Datasets

In [15]:
query_dataset_for_hub = DatasetDict.load_from_disk("./data/query-all-MiniLM-L6-v2-100-filters-embedded-results")

In [18]:
query_dataset_for_hub["train"].to_json("./hub/data/query-all-MiniLM-L6-v2-100-filters-embedded-results/train.jsonl", orient="records", lines=True)

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2312330

In [19]:
documents_dataset_for_hub = DatasetDict.load_from_disk("./data/corpus-all-MiniLM-L6-v2-50K-groups-multi-vector")

In [20]:
documents_dataset_for_hub["train"].to_json("./hub/data/corpus-all-MiniLM-L6-v2-50K-groups-multi-vector/train.jsonl", orient="records", lines=True)


Creating json from Arrow format:   0%|          | 0/50 [00:00<?, ?ba/s]

6369767346

## Try to load

In [ ]:
query_dataset_dict: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_test",
    name=None, 
    data_dir="query-all-MiniLM-L6-v2-100-filters-embedded-results"
)) 
query_dataset: Dataset = query_dataset_dict["train"]
query_dataset

In [ ]:
documents_dataset_dict: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_test",
    name=None, 
    data_dir="corpus-all-MiniLM-L6-v2-50K-groups-multi-vector"
)) 
documents: Dataset = query_dataset_dict["train"]
documents